Imports

In [73]:
import pandas as pd
import re
import sys
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error



Load Data

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists() and (PROJECT_ROOT.parent / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dubizzle_phones_clean.csv"
if RAW_DATA_PATH.exists():
    df = pd.read_csv(RAW_DATA_PATH)
else:
    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            f"Place dubizzle_phones_clean.csv at {RAW_DATA_PATH} or run this notebook in Google Colab and upload it."
        ) from exc
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No CSV file was uploaded.")
    uploaded_name = next(iter(uploaded))
    df = pd.read_csv(uploaded_name)


Saving dubizzle_phones_clean.csv to dubizzle_phones_clean.csv


In [74]:
import sys
import os
import importlib
target_path = '/content/DEPI-GRADUATION-PROJECT-task-2/src'

if os.path.exists(target_path):
    if target_path not in sys.path:
        sys.path.insert(0, target_path)
    else:
        sys.path.remove(target_path)
        sys.path.insert(0, target_path)
importlib.invalidate_caches()

try:
    from evaluation.simple_harness import metrics, add_to_log, save_log
    print("Evaluation imported successfully!")
except ModuleNotFoundError:
    sys.path.insert(0, '/content/DEPI-GRADUATION-PROJECT-task-2')
    importlib.invalidate_caches()
    from src.evaluation.simple_harness import metrics, add_to_log, save_log
    print("Evaluation imported successfully via src.evaluation!")

Evaluation imported successfully!


Dataset Shape

In [3]:
print("Dataset shape:", df.shape)

Dataset shape: (8213, 30)


Dataset Columns

In [4]:
print("Columns names:\n", df.columns.to_list())

Columns names:
 ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high']


In [5]:
print("Number of columns:", len(df.columns))

Number of columns: 30


In [6]:
df.head()

,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,is_parts_hint,accessory_first,bundle_hint,is_feature_phone_hint,price_placeholder,loc_area,loc_city,price_missing,price_suspicious,price_very_high
0,ايفون 13 iPhone,19500.0,مساكن المهندسين، مدينة نصر,/ad/%D8%A7%D9%8A%D9%81%D9%88%D9%86-13-iphone-I...,208854767,legacy,apple,NaN,NaN,NaN,...,False,False,False,False,False,مساكن المهندسين,مدينة نصر,False,False,False
1,Honor Magic V5,80000.0,اوبن اير مول، مدينتي,/ad/honor-magic-v5-ID209010621.html,209010621,legacy,honor,NaN,NaN,NaN,...,False,False,False,False,False,اوبن اير مول,مدينتي,False,False,False
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,57999.0,سموحة، الإسكندرية,/ad/oppo-find-x9-pro-%D8%B6%D9%85%D8%A7%D9%86-...,209003975,legacy,oppo,NaN,NaN,NaN,...,False,False,False,False,False,سموحة,الإسكندرية,False,False,False
3,Honor Robot Phone 512G 12G ram,94000.0,مدينة نصر، القاهرة,/ad/honor-robot-phone-512g-12g-ram-ID208841945...,208841945,legacy,honor,512.0,12.0,L1:unit,...,False,False,False,False,False,مدينة نصر,القاهرة,False,False,False
4,شاومي 17 الترا xiaomi 17 ultra,70000.0,التجمع الخامس، القاهرة الجديدة,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-17-%D8%A7%D...,208281312,legacy,xiaomi,NaN,NaN,NaN,...,False,False,False,False,False,التجمع الخامس,القاهرة الجديدة,False,False,False


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8213 entries, 0 to 8212
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  8213 non-null   object 
 1   price                  8213 non-null   float64
 2   location               8207 non-null   object 
 3   ad_url                 8213 non-null   object 
 4   ad_id                  8213 non-null   int64  
 5   run_id                 8213 non-null   object 
 6   brand                  7772 non-null   object 
 7   storage_gb             586 non-null    float64
 8   ram_gb                 177 non-null    float64
 9   storage_src            586 non-null    object 
 10  condition              439 non-null    object 
 11  condition_src          439 non-null    object 
 12  ram_src                177 non-null    object 
 13  has_5g                 8213 non-null   bool   
 14  has_warranty           8213 non-null   bool   
 15  has_

In [8]:
df.isna().sum().to_frame("Missing Values")

,Missing Values
title,0
price,0
location,6
ad_url,0
ad_id,0
run_id,0
brand,441
storage_gb,7627
ram_gb,8036
storage_src,7627


Data Types and Missing Values

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8213 entries, 0 to 8212
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  8213 non-null   object 
 1   price                  8213 non-null   float64
 2   location               8207 non-null   object 
 3   ad_url                 8213 non-null   object 
 4   ad_id                  8213 non-null   int64  
 5   run_id                 8213 non-null   object 
 6   brand                  7772 non-null   object 
 7   storage_gb             586 non-null    float64
 8   ram_gb                 177 non-null    float64
 9   storage_src            586 non-null    object 
 10  condition              439 non-null    object 
 11  condition_src          439 non-null    object 
 12  ram_src                177 non-null    object 
 13  has_5g                 8213 non-null   bool   
 14  has_warranty           8213 non-null   bool   
 15  has_

In [10]:
df.isna().sum().to_frame("Number of Missing Values")

,Number of Missing Values
title,0
price,0
location,6
ad_url,0
ad_id,0
run_id,0
brand,441
storage_gb,7627
ram_gb,8036
storage_src,7627


Check Duplicate Titles

In [11]:
df["title"].duplicated().sum()

np.int64(4513)

Inspect Duplicate Titles

In [12]:
df.loc[df["title"].duplicated(keep = False) , "title"].head(20).to_frame()

,title
0,ايفون 13 iPhone
1,Honor Magic V5
5,Apple - iPhone 17 Pro
9,آبل - آيفون 15 برو ماكس
14,آبل - آيفون 17 برو ماكس
15,آبل - آيفون 13 برو ماكس
17,Apple - iPhone 15
18,Samsung S25 Ultra
19,سامسونج اس 24 الترا
21,Apple - iPhone 17 Pro Max


Inspect Title Examples

In [13]:
df["title"].head(30)

,title
0,ايفون 13 iPhone
1,Honor Magic V5
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل
3,Honor Robot Phone 512G 12G ram
4,شاومي 17 الترا xiaomi 17 ultra
5,Apple - iPhone 17 Pro
6,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...
7,Honor 500 pro
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...
9,آبل - آيفون 15 برو ماكس


Check Title Normalization Examples

In [14]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False), "title"]

,title
21,Apple - iPhone 17 Pro Max
26,Apple - iPhone 17 Pro Max 512 GB
39,Apple - iPhone 17 Pro Max - Battery 98%
42,Apple - iPhone 17 Pro Max battery 100%
43,Apple iPhone 17 Pro Max (512GB) كسر زيرو | خال...
...,...
8049,Apple - iPhone 17 Pro Max
8057,Apple - iPhone 17 Pro Max
8072,Apple - iPhone 17 Pro Max battery 98% still in...
8084,Apple - iPhone 17 Pro Max


Check Title Variations

In [15]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False),"title"].nunique()

46

Inspect Unique Title Variations

In [16]:
df.loc[df["title"].str.contains("iPhone 17 Pro Max", case=False),"title"].drop_duplicates().to_list()

['Apple - iPhone 17 Pro Max',
 'Apple - iPhone 17 Pro Max 512 GB',
 'Apple - iPhone 17 Pro Max - Battery 98%',
 'Apple - iPhone 17 Pro Max battery 100%',
 'Apple iPhone 17 Pro Max (512GB) كسر زيرو | خالص ضريبة',
 'iPhone 17 pro max',
 'Apple - iPhone 17 Pro Max First high copy',
 'Apple - iPhone 17 Pro Max -Used Like New',
 'آبل - آيفون 17 برو ماكس iphone 17 pro max',
 'Apple - iPhone 17 Pro max',
 'iPhone 17 Pro Max 512gb',
 'iPhone 17 Pro Max 256GB – Silver',
 'iPhone 17 pro max 256 سيلفر',
 'IPhone 17 Pro Max silver zero scratches',
 'آيفون 17 برو ماكس  سيلفر - Apple iPhone 17 Pro Max siliver',
 'آبل IPhone 17 Pro max',
 'Apple - iPhone 17 Pro Max خربوش بسيط جدا جدا ف الباغه',
 'Apple - iPhone 17 Pro Max tradeline جديد متبرشم معفي ضريبة',
 'IPhone 17 pro max',
 'Iphone 17 pro max blue new',
 'iPhone 17 Pro Max 256 شريحه نسخه محلي مسجل بالعلبه فقط',
 'iphone 17 pro max',
 'iPhone 17 Pro Max كسر زيرو بدون خربوش للبيع',
 'iPhone 17 Pro Max 256GB Deep Blue – 100% Battery',
 'Apple - iPh

Title Normalization

In [17]:
df["title_normalized"] = df["title"].str.lower().str.strip()

In [18]:
df[["title", "title_normalized"]].head(20)

,title,title_normalized
0,ايفون 13 iPhone,ايفون 13 iphone
1,Honor Magic V5,honor magic v5
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,oppo find x9 pro ضمان محلي للبيع او البدل
3,Honor Robot Phone 512G 12G ram,honor robot phone 512g 12g ram
4,شاومي 17 الترا xiaomi 17 ultra,شاومي 17 الترا xiaomi 17 ultra
5,Apple - iPhone 17 Pro,apple - iphone 17 pro
6,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb_ايفون 15 برو_زيرو بسعر...
7,Honor 500 pro,honor 500 pro
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...
9,آبل - آيفون 15 برو ماكس,آبل - آيفون 15 برو ماكس


Compare Unique Titles

In [19]:
print("Original unique titles:", df["title"].nunique())
print("Normalized unique titles:", df["title_normalized"].nunique())

Original unique titles: 3700
Normalized unique titles: 3565


Normalize Arabic Letter Variants

In [20]:
df["title_normalized"] = (df["title_normalized"]
    .str.replace("أ" , "ا", regex=False)
    .str.replace("إ" , "ا", regex=False)
    .str.replace("آ" , "ا", regex=False)
    .str.replace("ى" , "ي", regex=False)
)

Compare Normalized Titles

In [21]:
df["title_normalized"].nunique()

3551

Inspect Title Punctuation

In [22]:
df["title_normalized"].head(20).to_list()

['ايفون 13 iphone',
 'honor magic v5',
 'oppo find x9 pro ضمان محلي للبيع او البدل',
 'honor robot phone 512g 12g ram',
 'شاومي 17 الترا xiaomi 17 ultra',
 'apple - iphone 17 pro',
 'للبيع iphone 15 pro 1tb_ايفون 15 برو_زيرو بسعر لقطة لسرعه البيع ممتاز',
 'honor 500 pro',
 'ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه للجادين للسرعه البيع 64 gb',
 'ابل - ايفون 15 برو ماكس',
 'huawei nova 15 max',
 'ايفون 16e جديد وارد الامارات مدفوع ضريبة | iphone 16e phone new',
 'oppo find x5 5g dual sim 256/8g black بحالة ممتازة',
 'samsung s24 ultra زيرو 256gb تيتانيوم مدفوع الضريبة بالضمان',
 'ابل - ايفون 17 برو ماكس',
 'ابل - ايفون 13 برو ماكس',
 'honor magic v5 dual sim 512/16g white gold جديد بضمان الوكيل',
 'apple - iphone 15',
 'samsung s25 ultra',
 'سامسونج اس 24 الترا']

Punctuation Normalization

In [23]:
df["title_punct_normalized"] = (
    df["title_normalized"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.strip()
    .str.split()
    .str.join(" ")
)

Compare Titles After Punctuation Normalization

In [24]:
print("Before:", df["title_normalized"].nunique())
print("After:", df["title_punct_normalized"].nunique())

Before: 3551
After: 3531


Review Normalized Titles

In [25]:
df[["title", "title_normalized", "title_punct_normalized"]].head(10)

,title,title_normalized,title_punct_normalized
0,ايفون 13 iPhone,ايفون 13 iphone,ايفون 13 iphone
1,Honor Magic V5,honor magic v5,honor magic v5
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,oppo find x9 pro ضمان محلي للبيع او البدل,oppo find x9 pro ضمان محلي للبيع او البدل
3,Honor Robot Phone 512G 12G ram,honor robot phone 512g 12g ram,honor robot phone 512g 12g ram
4,شاومي 17 الترا xiaomi 17 ultra,شاومي 17 الترا xiaomi 17 ultra,شاومي 17 الترا xiaomi 17 ultra
5,Apple - iPhone 17 Pro,apple - iphone 17 pro,apple iphone 17 pro
6,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb ايفون 15 برو زيرو بسعر...
7,Honor 500 pro,honor 500 pro,honor 500 pro
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) سعر لقطه للجا...
9,آبل - آيفون 15 برو ماكس,ابل - ايفون 15 برو ماكس,ابل ايفون 15 برو ماكس


Improve Punctuation Normalization

In [26]:
df["title_punct_normalized"] = (
    df["title_normalized"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

In [27]:
df.loc[df["title_normalized"] != df["title_punct_normalized"],["title_normalized", "title_punct_normalized"]].head(10)

,title_normalized,title_punct_normalized
5,apple - iphone 17 pro,apple iphone 17 pro
6,للبيع iphone 15 pro 1tb_ايفون 15 برو_زيرو بسعر...,للبيع iphone 15 pro 1tb ايفون 15 برو زيرو بسعر...
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) سعر لقطه لل...
9,ابل - ايفون 15 برو ماكس,ابل ايفون 15 برو ماكس
14,ابل - ايفون 17 برو ماكس,ابل ايفون 17 برو ماكس
15,ابل - ايفون 13 برو ماكس,ابل ايفون 13 برو ماكس
17,apple - iphone 15,apple iphone 15
21,apple - iphone 17 pro max,apple iphone 17 pro max
22,apple - iphone 14,apple iphone 14
26,apple - iphone 17 pro max 512 gb,apple iphone 17 pro max 512 gb


In [28]:
print("Before:", df["title_normalized"].nunique())
print("After:", df["title_punct_normalized"].nunique())

Before: 3551
After: 3551


Normalize Extra Spaces

In [29]:
df["title_punct_normalized"] = (
    df["title_punct_normalized"]
    .str.split()
    .str.join(" ")
)

In [30]:
df["title_punct_normalized"].head(20).to_list()

['ايفون 13 iphone',
 'honor magic v5',
 'oppo find x9 pro ضمان محلي للبيع او البدل',
 'honor robot phone 512g 12g ram',
 'شاومي 17 الترا xiaomi 17 ultra',
 'apple iphone 17 pro',
 'للبيع iphone 15 pro 1tb ايفون 15 برو زيرو بسعر لقطة لسرعه البيع ممتاز',
 'honor 500 pro',
 'ايفون 11 بحاله الزيرو (iphone11) سعر لقطه للجادين للسرعه البيع 64 gb',
 'ابل ايفون 15 برو ماكس',
 'huawei nova 15 max',
 'ايفون 16e جديد وارد الامارات مدفوع ضريبة | iphone 16e phone new',
 'oppo find x5 5g dual sim 256/8g black بحالة ممتازة',
 'samsung s24 ultra زيرو 256gb تيتانيوم مدفوع الضريبة بالضمان',
 'ابل ايفون 17 برو ماكس',
 'ابل ايفون 13 برو ماكس',
 'honor magic v5 dual sim 512/16g white gold جديد بضمان الوكيل',
 'apple iphone 15',
 'samsung s25 ultra',
 'سامسونج اس 24 الترا']

Inspect Remaining Punctuation

In [31]:
df.loc[df["title_punct_normalized"].str.contains(r"[|()]", regex=True, na=False), ["title", "title_punct_normalized"]].head(10)

,title,title_punct_normalized
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,ايفون 11 بحاله الزيرو (iphone11) سعر لقطه للجا...
11,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iPh...,ايفون 16e جديد وارد الامارات مدفوع ضريبة | iph...
40,موبايل Samsung S23 Ultra مستعمل (256GB / 12GB ...,موبايل samsung s23 ultra مستعمل (256gb / 12gb ...
43,Apple iPhone 17 Pro Max (512GB) كسر زيرو | خال...,apple iphone 17 pro max (512gb) كسر زيرو | خال...
72,Infinix smart 20 (4+4/64) جديد,infinix smart 20 (4+4/64) جديد
196,ايفون ١٦ بلس (iphone 16 plus),ايفون ١٦ بلس (iphone 16 plus)
327,Apple iPhone 16 Pro (128 g) كسر زيرو | بطارية ...,apple iphone 16 pro (128 g) كسر زيرو | بطارية ...
527,Xiaomi Poco X3 (128/6),xiaomi poco x3 (128/6)
529,جهازين 11pro max (X),جهازين 11pro max (x)
653,iPhone 12 Pro Max (256) zero tax,iphone 12 pro max (256) zero tax


Normalize Arabic and English Digits

In [32]:
digit_map = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

df["title_normalized"] = (df["title_normalized"].str.translate(digit_map))

Remove Arabic Diacritics

In [33]:
df["title_normalized"] = (df["title_normalized"].str.replace(r"[\u064B-\u065F\u0670]", "" , regex=True))

In [34]:
df.loc[df["title"].str.contains("[ًٌٍَُِّْ]", na=False),["title", "title_normalized"]].head(10)

,title,title_normalized
354,سامسونج جالاكسي A07 مساحة 128GB رام 6GB - زير...,سامسونج جالاكسي a07 مساحة 128gb رام 6gb - زير...
507,Realme 9i 6/128 بحالة ممتازة جدًا + الكرتونة,realme 9i 6/128 بحالة ممتازة جدا + الكرتونة
746,للبيغ Infinix Smart بحالة ممتازة وبسعر لُقطة!,للبيغ infinix smart بحالة ممتازة وبسعر لقطة!
1499,لقطه العمر هاتف سامسونج جالاكسي ايه 13 ب 3000ج...,لقطه العمر هاتف سامسونج جالاكسي ايه 13 ب 3000ج...
2296,13 عاديً,13 عادي
2443,اوبو رينو 6 5g الفون زي الجديد حرفياً موضح بالصور,اوبو رينو 6 5g الفون زي الجديد حرفيا موضح بالصور
2495,​ فرصة مميزة للبيع: هاتف Infinix Note 50s كسر ...,​ فرصة مميزة للبيع: هاتف infinix note 50s كسر ...
3507,آبل - آيفون 17 برو ماكس زيرو تماماً جديد,ابل - ايفون 17 برو ماكس زيرو تماما جديد
3790,موبايل مختلف وفخم جدًا لعشاق التميز OPPO Find ...,موبايل مختلف وفخم جدا لعشاق التميز oppo find n...
4020,موبل سامسونج ِ30 جيد جدا للبيع,موبل سامسونج 30 جيد جدا للبيع


Compare Unique Titles Before and After Punctuation Normalization

In [35]:
print("Unique normalized titles:", df["title_normalized"].nunique())
print("Unique punctuation-normalized titles:", df["title_punct_normalized"].nunique())

Unique normalized titles: 3521
Unique punctuation-normalized titles: 3531


Rebuild Punctuation-Normalized Titles

In [36]:
df["title_punct_normalized"] = (df["title_normalized"]
    .str.replace(r"[-_]", " ", regex=True)
    .str.split()
    .str.join(" ")
)

In [37]:
df[["title", "brand", "storage_gb", "ram_gb"]].head(20)

,title,brand,storage_gb,ram_gb
0,ايفون 13 iPhone,apple,NaN,NaN
1,Honor Magic V5,honor,NaN,NaN
2,OPPO Find X9 Pro ضمان محلي للبيع او البدل,oppo,NaN,NaN
3,Honor Robot Phone 512G 12G ram,honor,512.0,12.0
4,شاومي 17 الترا xiaomi 17 ultra,xiaomi,NaN,NaN
5,Apple - iPhone 17 Pro,apple,NaN,NaN
6,للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر...,apple,1024.0,NaN
7,Honor 500 pro,honor,NaN,NaN
8,ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه لل...,apple,64.0,NaN
9,آبل - آيفون 15 برو ماكس,apple,NaN,NaN


In [38]:
print("Unique normalized titles:", df["title_normalized"].nunique())
print("Unique punctuation-normalized titles:", df["title_punct_normalized"].nunique())

Unique normalized titles: 3521
Unique punctuation-normalized titles: 3500


In [39]:
df[["title", "title_normalized", "title_punct_normalized"]].sample(10, random_state=42)

,title,title_normalized,title_punct_normalized
6791,فيفو x200 fe,فيفو x200 fe,فيفو x200 fe
7863,اوبو رينو 10 برو,اوبو رينو 10 برو,اوبو رينو 10 برو
3647,شاومي موديل اخر,شاومي موديل اخر,شاومي موديل اخر
7364,Apple - iPhone 17 Pro Max,apple - iphone 17 pro max,apple iphone 17 pro max
5549,آبل - آيفون 14 برو,ابل - ايفون 14 برو,ابل ايفون 14 برو
5584,آبل - آيفون 13,ابل - ايفون 13,ابل ايفون 13
2540,Apple - iPhone 15 Pro Max,apple - iphone 15 pro max,apple iphone 15 pro max
1882,Vivo y19 s Pro,vivo y19 s pro,vivo y19 s pro
970,آبل - آيفون 11 برو ماكس,ابل - ايفون 11 برو ماكس,ابل ايفون 11 برو ماكس
5952,Apple - iPhone 12 Pro Max,apple - iphone 12 pro max,apple iphone 12 pro max


Check for Extra Spaces in Normalized Titles

In [40]:
df["title_punct_normalized"].str.contains(r"\s{2,}", regex=True, na=False).sum()

np.int64(0)

 Verify Final Title Normalization

In [41]:
df[["title", "title_normalized", "title_punct_normalized"]].sample(10, random_state=42)

,title,title_normalized,title_punct_normalized
6791,فيفو x200 fe,فيفو x200 fe,فيفو x200 fe
7863,اوبو رينو 10 برو,اوبو رينو 10 برو,اوبو رينو 10 برو
3647,شاومي موديل اخر,شاومي موديل اخر,شاومي موديل اخر
7364,Apple - iPhone 17 Pro Max,apple - iphone 17 pro max,apple iphone 17 pro max
5549,آبل - آيفون 14 برو,ابل - ايفون 14 برو,ابل ايفون 14 برو
5584,آبل - آيفون 13,ابل - ايفون 13,ابل ايفون 13
2540,Apple - iPhone 15 Pro Max,apple - iphone 15 pro max,apple iphone 15 pro max
1882,Vivo y19 s Pro,vivo y19 s pro,vivo y19 s pro
970,آبل - آيفون 11 برو ماكس,ابل - ايفون 11 برو ماكس,ابل ايفون 11 برو ماكس
5952,Apple - iPhone 12 Pro Max,apple - iphone 12 pro max,apple iphone 12 pro max


Count Unique Titles After Final Normalization

In [42]:
df["title_punct_normalized"].nunique()

3500

Create Final Normalized Title Groups

In [43]:
df["title_group"] = df["title_punct_normalized"]

In [44]:
print("Missing groups:", df["title_group"].isna().sum())
print("Total groups:", df["title_group"].nunique())
print("Total rows:", len(df))

Missing groups: 0
Total groups: 3500
Total rows: 8213


Validate Title Normalization

In [45]:
print("Original unique titles:", df["title"].nunique())
print("Normalized unique titles:", df["title_normalized"].nunique())
print("Final title groups:", df["title_group"].nunique())
print("Missing groups:", df["title_group"].isna().sum())
print("Total rows:", len(df))

Original unique titles: 3700
Normalized unique titles: 3521
Final title groups: 3500
Missing groups: 0
Total rows: 8213


# Grouped Train-Test Split

In [75]:
from sklearn.model_selection import GroupShuffleSplit
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["title_group"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

Validate Grouped Split

In [76]:
train_groups = set(train_df["title_group"])
test_groups = set(test_df["title_group"])

overlap = train_groups.intersection(test_groups)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Overlapping groups:", len(overlap))

Train rows: 6417
Test rows: 1796
Overlapping groups: 0


Save Frozen Test Set

In [77]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

test_df.to_csv(output_dir / "frozen_test_set.csv", index=False)


In [78]:
print("Train groups:", train_df["title_group"].nunique())
print("Test groups:", test_df["title_group"].nunique())
print("Total groups:", df["title_group"].nunique())

Train groups: 2800
Test groups: 700
Total groups: 3500


In [79]:
print("Development rows:", len(train_df))
print("Frozen test rows:", len(test_df))
print("Frozen test file exists:", (output_dir / "frozen_test_set.csv").exists())

Development rows: 6417
Frozen test rows: 1796
Frozen test file exists: True


##### GroupKFold Cross-Validation

In [80]:

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(
        train_df,
        groups=train_df["title_group"]
    ),
    start=1
):
    print(
        f"Fold {fold}: "
        f"Train={len(train_idx)}, "
        f"Validation={len(val_idx)}"
    )

Fold 1: Train=5133, Validation=1284
Fold 2: Train=5133, Validation=1284
Fold 3: Train=5134, Validation=1283
Fold 4: Train=5134, Validation=1283
Fold 5: Train=5134, Validation=1283


In [81]:
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df,groups=train_df["title_group"]),
    start=1
):
    train_groups = set(train_df.iloc[train_idx]["title_group"])
    val_groups = set(train_df.iloc[val_idx]["title_group"])

    overlap = train_groups.intersection(val_groups)

    print(f"Fold {fold}: Overlapping groups = {len(overlap)}")

Fold 1: Overlapping groups = 0
Fold 2: Overlapping groups = 0
Fold 3: Overlapping groups = 0
Fold 4: Overlapping groups = 0
Fold 5: Overlapping groups = 0


frozen to Data

In [82]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        groups=train_df["title_group"]
    )
)

train_part = train_df.iloc[train_idx].copy()
val_df = train_df.iloc[val_idx].copy()

print("Train rows:", len(train_part))
print("Validation rows:", len(val_df))

Train rows: 5165
Validation rows: 1252


In [83]:
split_dir = PROJECT_ROOT / "data" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)
train_part.to_csv(output_dir / "train_set.csv", index=False)
val_df.to_csv(split_dir / "validation.csv", index=False)

In [84]:
train_part_groups = set(train_part["title_group"])
val_part_groups = set(val_df["title_group"])
test_part_groups = set(test_df["title_group"])

print("Saved train rows:", len(train_part))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("Train/validation overlap:", len(train_part_groups & val_part_groups))
print("Train/test overlap:", len(train_part_groups & test_part_groups))
print("Validation/test overlap:", len(val_part_groups & test_part_groups))

assert len(train_part_groups & val_part_groups) == 0
assert len(train_part_groups & test_part_groups) == 0
assert len(val_part_groups & test_part_groups) == 0

print("All splits are group-separated!")

Saved train rows: 5165
Validation rows: 1252
Test rows: 1796
Train/validation overlap: 0
Train/test overlap: 0
Validation/test overlap: 0
All splits are group-separated!


-----------------------------------------------------------

baseline

What is baseline ?

In [85]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

In [86]:
def grouped_median_baseline(data, n_splits=5):
    predictions = data.copy()

    # Setting up arrays to hold our OOF predictions
    global_preds = np.full(len(data), np.nan)
    brand_preds = np.full(len(data), np.nan)

    gkf = GroupKFold(n_splits=n_splits)

    # Let's loop through the splits
    for train_idx, val_idx in gkf.split(data, groups=data["title_group"]):
        train_df = data.iloc[train_idx]
        val_df = data.iloc[val_idx]

        # just a sanity check to make sure group kfold is actually working as intended
        overlap = set(train_df["title_group"]) & set(val_df["title_group"])
        assert not overlap, f"Group leakage detected: {len(overlap)} overlapping groups"

        # calc baseline stats on train fold only!! very important to avoid leakage
        global_med = train_df["price"].median()

        # grouping by brand to find median price per brand
        brand_meds = train_df.groupby("brand")["price"].median()

        # Fill in global predictions
        global_preds[val_idx] = global_med

        # map the brand medians, and if a brand wasn't in train, fallback to global median
        mapped_brands = val_df["brand"].map(brand_meds)
        filled_brands = mapped_brands.fillna(global_med)

        brand_preds[val_idx] = filled_brands.to_numpy()

    # save back into df
    predictions["pred_E0"] = global_preds
    predictions["pred_E1"] = brand_preds

    # calculate final MAE
    # Note: remember we are evaluating on the whole thing (OOF predictions combined)
    mae_global = mean_absolute_error(data["price"], predictions["pred_E0"])
    mae_brand = mean_absolute_error(data["price"], predictions["pred_E1"])

    scores = {
        "global_median_mae": mae_global,
        "brand_median_mae": mae_brand,
    }

    return predictions, scores

In [87]:
baseline_predictions, baseline_scores = grouped_median_baseline(train_part)

In [88]:
print("Global median MAE:", baseline_scores["global_median_mae"])
print("Brand median MAE:", baseline_scores["brand_median_mae"])

Global median MAE: 16848.07357212004
Brand median MAE: 15563.937270087124


In [89]:
src_path = str(PROJECT_ROOT / "DEPI-GRADUATION-PROJECT-task-2" / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from evaluation.simple_harness import metrics, add_to_log, save_log

In [90]:
val_predictions = val_df.copy()

global_med = train_part["price"].median()
brand_meds = train_part.groupby("brand")["price"].median()

val_predictions["pred_E0"] = global_med
val_predictions["pred_E1"] = (
    val_predictions["brand"].map(brand_meds).fillna(global_med)
)

log = []

mae, medae = metrics(val_predictions, "pred_E0")
add_to_log(log, "E0", "global median", mae, medae)

mae, medae = metrics(val_predictions, "pred_E1")
add_to_log(log, "E1", "brand median", mae, medae)

log_path = save_log(log, output_dir / "experiment_log.csv")

print("Experiment log saved:", log_path)

Overall -> MAE: 15422.15 | MedAE: 10500.0

By brand:
                  mean   median  count
_brand                                
apple     18744.090717  13000.0    474
google    17000.000000  17000.0      1
honor     11709.000000   7250.0     50
huawei    19446.153846  12000.0     26
infinix    7011.500000   7750.0     26
itel       6833.333333   6500.0      3
lenovo    11850.000000  11850.0      2
motorola   5520.000000   2500.0      5
nokia      2000.000000   2000.0      1
oneplus   15642.857143  22500.0      7
oppo      11305.970149  11000.0     67
realme     9314.200000   7800.0     55
samsung   16404.166154  10000.0    325
sony      15010.500000  13249.5      4
tecno      5150.000000   5150.0      2
unknown   14946.478873  10500.0     71
vivo      12091.625000   7500.0     24
xiaomi     8479.703704   7750.0    108
zte       11500.000000  11500.0      1

By price band:
                    mean   median  count
price_band                              
cheap        9425.541147   950

In [91]:
print(df["price"].describe())
print(df["price"].quantile([0.33, 0.66]))

count      8213.000000
mean      24329.751005
std       22784.033249
min        1000.000000
25%        7500.000000
50%       16000.000000
75%       35000.000000
max      185500.000000
Name: price, dtype: float64
0.33    10000.0
0.66    25500.0
Name: price, dtype: float64


In [92]:
import zipfile
from google.colab import files

zip_path = "/content/project_splits.zip"

files_to_save = [
    "data/processed/train_set.csv",
    "data/processed/frozen_test_set.csv",
    "data/processed/experiment_log.csv",
    "data/splits/validation.csv"
]

with zipfile.ZipFile(zip_path, "w") as z:
    for file_path in files_to_save:
        z.write(file_path, arcname=file_path)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [93]:
print("Train rows:", len(train_part))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

Train rows: 5165
Validation rows: 1252
Test rows: 1796


In [94]:
print(train_part.shape)

(5165, 33)


In [95]:
train_part.columns.tolist()

['title',
 'price',
 'location',
 'ad_url',
 'ad_id',
 'run_id',
 'brand',
 'storage_gb',
 'ram_gb',
 'storage_src',
 'condition',
 'condition_src',
 'ram_src',
 'has_5g',
 'has_warranty',
 'has_box',
 'mentions_exchange',
 'is_wanted_ad',
 'is_accessory_hint',
 'is_tablet_hint',
 'is_parts_hint',
 'accessory_first',
 'bundle_hint',
 'is_feature_phone_hint',
 'price_placeholder',
 'loc_area',
 'loc_city',
 'price_missing',
 'price_suspicious',
 'price_very_high',
 'title_normalized',
 'title_punct_normalized',
 'title_group']

Imports